# Agents and workflows

This notebook works through the agentic patterns described in Anthropic's [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) article.  Each section implements one pattern end-to-end so you can step through it cell by cell.

Patterns covered (added as the course module progresses):

- **Evaluator-Optimizer** — a generator and a critic loop until the critic accepts the output.
- **Parallelization** — split one complex task into specialized sub-tasks that run concurrently, then aggregate.
- **Chaining** — feed each step's output into the next as a focused, validated handoff.
- **Routing** — classify the request, then dispatch it to a specialist prompt built for that category.


In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

In [ ]:
def _add_message(messages: list, role: str, text: str) -> None:
    messages.append({"role": role, "content": text})

def add_user_message(messages: list, text: str) -> None:
    _add_message(messages, "user", text)

def add_assistant_message(messages: list, text: str) -> None:
    _add_message(messages, "assistant", text)

def chat(messages: list, system_prompt: str | None = None, temperature: float = 1.0, stop: list | None = None) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system_prompt is not None:
        params["system"] = system_prompt
    if stop is not None:
        params["stop_sequences"] = stop
    response = client.messages.create(**params)
    return response.content[0].text

# Evaluator-Optimizer

The Evaluator-Optimizer pattern splits a task across two roles that take turns:

- **Optimizer** — generates a candidate answer.  On each iteration after the first it also sees the previous attempt and the evaluator's feedback, and tries to address it.
- **Evaluator** — grades the candidate against a fixed rubric and either accepts it or returns concrete, actionable feedback for the next round.

The loop stops when the evaluator accepts the answer, or when a max-iteration cap is reached so a stubborn disagreement doesn't run up your bill.

This pattern shines when (a) a single-shot generation is too unreliable to trust, and (b) "good" is easy to describe in a checklist but hard to satisfy in one pass.  Examples: translation, code refactoring, structured-data extraction with strict schemas, marketing copy with brand constraints.

The example below is intentionally small and visual: the optimizer writes a tagline for a new product, and the evaluator checks it against a short rubric (length, no buzzwords, complete thought, benefit-led).

In [ ]:
OPTIMIZER_SYSTEM = """You are a senior copywriter writing taglines for new products.

When you are given a product brief, return ONLY the tagline itself — no quotes, no commentary, no preamble.

If you are also given feedback from a previous attempt, treat it as a binding rewrite brief: address every point the editor raised, and do not repeat any previous tagline verbatim."""

EVALUATOR_SYSTEM = """You are a discerning brand editor reviewing a tagline against a fixed rubric.

Rubric (every rule must pass for ACCEPT):
1. Seven words or fewer.
2. Contains none of these buzzwords: revolutionary, innovative, next-gen, cutting-edge, world-class, game-changing, seamless.
3. Reads as a complete thought — not a fragment that only makes sense alongside the product name.
4. Evokes the product's core benefit, not just its category.

Respond in exactly this format and nothing else:
VERDICT: <ACCEPT or REVISE>
FEEDBACK: <one or two sentences. On ACCEPT, briefly say why it works. On REVISE, name the specific rule(s) violated and what to change.>"""

In [ ]:
def generate_tagline(brief: str, previous_attempt: str | None = None, feedback: str | None = None) -> str:
    messages = []
    user_msg = f"Product brief:\n{brief}"
    if previous_attempt is not None:
        user_msg += f"\n\nPrevious attempt:\n{previous_attempt}\n\nEditor feedback:\n{feedback}"
    add_user_message(messages, user_msg)
    return chat(messages, system_prompt=OPTIMIZER_SYSTEM, temperature=1.0).strip()

In [ ]:
def evaluate_tagline(brief: str, tagline: str) -> tuple[str, str]:
    messages = []
    add_user_message(messages, f"Product brief:\n{brief}\n\nCandidate tagline:\n{tagline}")
    raw = chat(messages, system_prompt=EVALUATOR_SYSTEM, temperature=0.0)

    verdict = "REVISE"
    feedback = raw.strip()
    for line in raw.splitlines():
        upper = line.upper()
        if upper.startswith("VERDICT:"):
            verdict = line.split(":", 1)[1].strip().upper()
        elif upper.startswith("FEEDBACK:"):
            feedback = line.split(":", 1)[1].strip()
    return verdict, feedback

In [ ]:
def evaluator_optimizer(brief: str, max_iterations: int = 4) -> str:
    tagline = generate_tagline(brief)
    print(f"Iteration 1 candidate: {tagline}")

    for i in range(2, max_iterations + 1):
        verdict, feedback = evaluate_tagline(brief, tagline)
        print(f"  evaluator: {verdict} — {feedback}")
        if verdict == "ACCEPT":
            return tagline
        tagline = generate_tagline(brief, previous_attempt=tagline, feedback=feedback)
        print(f"Iteration {i} candidate: {tagline}")

    print("  (max iterations reached — returning last candidate)")
    return tagline

In [ ]:
brief = (
    "A noise-cancelling alarm clock that wakes you with light instead of sound, "
    "designed for shift workers and parents of newborns who need to wake up without "
    "disturbing anyone next to them."
)

final = evaluator_optimizer(brief)
print(f"\nFinal tagline: {final}")

## Things to try

- Tighten the rubric (e.g. require a verb in the imperative mood) and watch how many extra iterations the loop needs.
- Lower the optimizer's `temperature` and observe the candidates converging faster but exploring less.
- Swap the brief for one that resists the rubric (e.g. a brief whose obvious phrasings include a buzzword) and see whether the loop hits the iteration cap.
- Replace the optimizer's model with a smaller/cheaper one while keeping the evaluator on Sonnet — a common production pattern that trades a few extra iterations for lower per-token cost.

# Parallelization

The Parallelization pattern splits one complex prompt into several focused sub-prompts that run at the same time, then combines their outputs with a final aggregator call.  Two reasons to reach for it:

- **Latency** — running four two-second calls concurrently finishes in ~2 seconds total instead of ~8.
- **Quality** — each sub-prompt can be specialized (different system prompt, different rubric, even a different model) without the cost of one giant catch-all prompt that asks Claude to juggle every concern at once.

The sub-tasks do not need to be identical: each can have its own role, rubric, or tool set.  They just need to produce independent slices that the aggregator can combine.

In this example we triage a customer support ticket by running four specialized analyses in parallel:

- **Sentiment** — frustrated / neutral / pleased, with a one-line justification
- **Urgency** — low / medium / high / critical, anchored to the business impact mentioned in the ticket
- **Category** — billing / technical / account / feature-request / other
- **Resolution info** — what does the support agent need to look up before replying?

The aggregator then combines all four into a single routing decision.


In [ ]:
TICKET = """\
Subject: Charged twice for May, can't log in to check

Hi — I just noticed two charges of $49.99 on my card on May 14th, both from your company.  I've been a customer for three years, never had an issue before, but right now I'm pretty frustrated.

To make it worse, when I try to log in to my account to look at the billing history myself, I get "Session expired" immediately every time, even right after entering the password.  I've tried Chrome and Firefox, cleared cookies, same thing.

I have a board meeting on Friday and I was planning to use the export feature to prep my slides.  If I can't get in by Thursday morning I'm going to need to switch to a competitor for this week's reporting at least.

— Diana K.
"""

In [ ]:
SUBTASKS = {
    "sentiment": """Classify the emotional tone of the support ticket below in exactly one word from this set: frustrated, neutral, pleased.

On a second line, give a one-sentence justification quoting the strongest evidence from the ticket.  Do not output anything else.""",

    "urgency": """Assign an urgency level for the support ticket below from this set: low, medium, high, critical.

On a second line, give a one-sentence justification — name the specific deadline or business impact that drives the level.  Do not output anything else.""",

    "category": """Classify the support ticket below into exactly one primary category from this set: billing, technical, account, feature-request, other.  If multiple apply, choose the one most blocking the customer's stated goal.

On a second line, name the secondary category if one applies, or write "none".  Do not output anything else.""",

    "resolution_info": """The support ticket below is about to be handed to a support agent.  List the specific pieces of information the agent needs to look up before replying — account IDs, transaction records, log entries, customer history items, etc.

One bullet per item, three to six bullets.  Do not propose a reply.""",
}

In [ ]:
import concurrent.futures
import time

def run_subtask(label: str, system_prompt: str, ticket: str) -> tuple[str, str]:
    messages = []
    add_user_message(messages, ticket)
    return label, chat(messages, system_prompt=system_prompt, temperature=0.0)

def triage_parallel(ticket: str, subtasks: dict) -> dict:
    start = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(subtasks)) as pool:
        futures = [pool.submit(run_subtask, label, sys, ticket) for label, sys in subtasks.items()]
        results = {label: out for label, out in (f.result() for f in concurrent.futures.as_completed(futures))}
    elapsed = time.perf_counter() - start
    print(f"{len(subtasks)} sub-tasks completed in {elapsed:.1f}s\n")
    return results

In [ ]:
results = triage_parallel(TICKET, SUBTASKS)
for label in SUBTASKS:
    print(f"--- {label} ---\n{results[label]}\n")

In [ ]:
AGGREGATOR_SYSTEM = """You are a support triage lead.  You are given the original ticket and four independent analyses of it.  Produce a final routing decision.

Output exactly these four labeled lines and nothing else:
ROUTE_TO: <one of: billing-team, technical-team, account-team, retention-team>
PRIORITY: <P0, P1, P2, or P3>
LOOKUP_BEFORE_REPLY: <comma-separated short list of items the agent should fetch first>
ONE_LINE_RATIONALE: <one sentence — name the two analyses that most drove the routing and priority>"""

def aggregate(ticket: str, results: dict) -> str:
    payload = f"Original ticket:\n{ticket}\n\nAnalyses:\n"
    for label, output in results.items():
        payload += f"\n[{label}]\n{output}\n"
    messages = []
    add_user_message(messages, payload)
    return chat(messages, system_prompt=AGGREGATOR_SYSTEM, temperature=0.0)

print(aggregate(TICKET, results))

## Confirming the speedup is real

For comparison, run the same four sub-tasks one after another and watch the wall-clock difference.  The parallel run is bounded by the slowest single call; the sequential run is bounded by the sum.

In [ ]:
def triage_sequential(ticket: str, subtasks: dict) -> dict:
    start = time.perf_counter()
    results = {}
    for label, sys in subtasks.items():
        _, output = run_subtask(label, sys, ticket)
        results[label] = output
    elapsed = time.perf_counter() - start
    print(f"{len(subtasks)} sub-tasks completed sequentially in {elapsed:.1f}s")
    return results

_ = triage_sequential(TICKET, SUBTASKS)

## Things to try

- Add a fifth sub-task that calls a cheaper model (e.g. Haiku) for the simpler dimensions, leaving Sonnet on the hard ones — the thread pool does not care that the workers are heterogeneous.
- Replace `ThreadPoolExecutor` with `asyncio.gather` over `AsyncAnthropic` for an event-loop-friendly version that scales further when the sub-task count is large.
- Reuse the pattern for long-document review: split the document into chunks, run a specialized analysis per chunk in parallel, then aggregate the chunk-level findings into a document-level summary.
- Swap the ticket for one that is genuinely ambiguous (e.g. a feature request worded as a complaint) and observe how the aggregator weighs the conflicting analyses.

# Chaining

The Chaining pattern feeds the output of one Claude call into the next as a focused, single-purpose handoff.  Compared to a single mega-prompt that tries to do everything at once, chaining gives you:

- **Focused prompts** — each step addresses one concern (extract / transform / compose) and is easier for the model to follow reliably.  This is the practical answer to "Claude is dropping constraint #4 in my long prompt": split the prompt so #4 has its own call.
- **Inspectable intermediates** — you can print, log, or assert on the data between steps.  That visibility is impossible inside a single prompt.
- **Validation between steps** — if the intermediate fails a schema check or a business rule, the chain can stop, retry just that step, or fall back.  Garbage doesn't propagate.
- **Cheap remixes** — once you have a structured intermediate, swapping the final composition step (email → Slack message → Jira ticket) is one prompt change.

The cost is more API calls in the happy path.  The benefit shows up the moment a "do everything" prompt starts silently ignoring constraints — which is exactly the failure mode chaining is designed to fix.

In this example we turn raw meeting notes into a follow-up email through three steps:

1. **Extract** — pull structured action items from the transcript as JSON.
2. **Enrich** — for each action item, draft a short context note explaining what is needed and why.
3. **Compose** — write a recap email that lists the enriched action items.

Between steps 1 and 2 we validate that the JSON parses and that every item carries the required keys — a real check that step 1 did its job before its output reaches step 2.


In [ ]:
NOTES = """\
Q3 Roadmap Sync — May 14.  Attendees: Lia (PM), Marcus (Eng Lead), Sara (Design), Tom (Support).

Lia kicked off by recapping last quarter's miss on the export feature — we shipped CSV but skipped XLSX, and Tom confirmed that's now the #2 complaint in the support queue.

Sara walked through the new dashboard mocks.  Everyone liked the sparkline treatment but agreed the empty state is too dense.  Sara said she'd revise the empty state by next Wednesday so front-end work can start without waiting on more design rounds.

Marcus flagged that the auth library upgrade has been sitting on his sprint board for three weeks.  The 0.x branch we depend on hits end-of-life June 30 and the migration involves a token format change, so it has to land before then.  He committed to spiking the migration by Friday and bringing a real estimate to the next sync.

Tom asked about the SSO rollout.  Lia said it's still gated on legal's review of the SAML metadata exchange.  She'll ping legal again on Monday and confirm whether the prospect demo on May 28 is still feasible — if legal slips, the demo either moves or runs without SSO.

Open question (no firm owner): do we keep the trial extension flow for self-serve, or kill it?  Marcus thinks it's quietly driving ~15% of paid conversions but the analytics are stale.  Lia will pull fresh numbers and bring them to the next meeting.

Next sync: May 21, same time.
"""

In [ ]:
import json as json_lib

EXTRACT_SYSTEM = """You extract action items from meeting notes.

Output ONLY a JSON array.  No prose, no markdown code fences, no explanation.

Each element must have exactly these keys:
- owner: string — the name of the person responsible, or "unassigned" if no one was named
- description: string — a concise verb-led task (e.g. "Revise the dashboard empty state mock")
- due_date: string — an ISO date YYYY-MM-DD if explicit in the notes, otherwise "TBD"
- source_quote: string — a short phrase from the notes that justifies this action item

Include only concrete commitments.  Skip topics that were discussed but resulted in no decision or owner."""

def extract_action_items(notes: str) -> list[dict]:
    messages = []
    add_user_message(messages, notes)
    raw = chat(messages, system_prompt=EXTRACT_SYSTEM, temperature=0.0).strip()
    if raw.startswith("```"):
        inner = raw.strip("`")
        if inner.lower().startswith("json"):
            inner = inner[4:]
        raw = inner.strip()
    return json_lib.loads(raw)

In [ ]:
items = extract_action_items(NOTES)

REQUIRED_KEYS = {"owner", "description", "due_date", "source_quote"}
assert len(items) >= 1, "Step 1 produced no action items — chain aborts."
for i, item in enumerate(items):
    missing = REQUIRED_KEYS - set(item)
    assert not missing, f"Item {i} missing keys {missing}: {item}"

print(f"Extracted {len(items)} action items.\n")
print(json_lib.dumps(items, indent=2))

In [ ]:
ENRICH_SYSTEM = """You write a short context note for the assignee of an action item.

Given the meeting context and a single action item, produce one or two sentences that explain why this task matters now and what the assignee needs to know to start work.

Do not restate the task itself, the owner, or the due date — those are already shown to the reader.  Output only the context note: no preamble, no labels, no quote marks."""

def enrich_item(item: dict, meeting_context: str) -> str:
    payload = (
        f"Meeting context:\n{meeting_context}\n\n"
        f"Action item:\n{json_lib.dumps(item, indent=2)}"
    )
    messages = []
    add_user_message(messages, payload)
    return chat(messages, system_prompt=ENRICH_SYSTEM, temperature=0.0).strip()

for item in items:
    item["context"] = enrich_item(item, NOTES)
    print(f"--- {item['owner']}: {item['description']} ---")
    print(item["context"])
    print()

In [ ]:
COMPOSE_SYSTEM = """You draft a meeting recap email.

Given the meeting context and a list of enriched action items, produce a professional email with:
- A brief one-paragraph recap of what was discussed.
- An "Action Items" section showing each item with its owner, task, due date, and context note, in that order.
- A one-line sign-off.

Use a warm but professional tone.  Do not invent agenda items that were not in the input.  Do not include a subject line."""

def compose_email(items: list[dict], meeting_context: str) -> str:
    payload = f"Meeting context:\n{meeting_context}\n\nEnriched action items:\n"
    for item in items:
        payload += (
            f"\n- Owner: {item['owner']}"
            f"\n  Task: {item['description']}"
            f"\n  Due: {item['due_date']}"
            f"\n  Context: {item['context']}\n"
        )
    messages = []
    add_user_message(messages, payload)
    return chat(messages, system_prompt=COMPOSE_SYSTEM, temperature=0.5)

print(compose_email(items, NOTES))

## Things to try

- Break step 1 deliberately by lowering the model's temperature *and* deleting the "No prose, no markdown code fences" line from `EXTRACT_SYSTEM`.  Watch the JSON validation in cell `extract_action_items` catch the regression before it poisons steps 2 and 3 — that is the point of validating between steps.
- Swap the final step's `COMPOSE_SYSTEM` for one that produces a Slack message or a Jira-ready ticket description.  Because the intermediate is structured, you only rewrite the last prompt, not the whole chain.
- Cram all three steps into a single mega-prompt as a comparison ("extract action items, write a context note for each, then compose a recap email").  Note which constraints get silently dropped — usually the structured-JSON part or the per-item context — and compare to the chained version on the same input.
- Run step 2 (`enrich_item`) inside a `ThreadPoolExecutor` from the Parallelization section.  The chain stays linear at the *step* level, but the per-item loop inside a step can fan out — the two patterns compose.

# Routing

The Routing pattern uses one cheap classification call to decide *which* specialist prompt should handle a request, then runs that specialist.  Instead of one mega-prompt that tries to be good at every kind of input ("explain science *and* tell history stories *and* write product reviews *and*..."), you keep N small, focused prompts and let the router pick.

Why route instead of cramming everything into a single prompt:

- **Per-category quality** — each specialist can have its own structure, tone, length, and even model.  A science explainer wants a hook–body–payoff arc; a how-to tutorial wants imperative-voice numbered steps.  These constraints fight each other inside one prompt.
- **Per-category cost** — the router can run on a cheap model (Haiku) while only the chosen specialist pays for Sonnet, since classification is almost always the easier sub-task.
- **Safe fallback** — if the router returns `unknown`, the system can refuse gracefully or ask for a reformulation, instead of hallucinating a half-correct answer.
- **Add a new category by adding one prompt** — no surgery on a shared mega-prompt, no regression risk for the existing categories.

In this example we generate short video scripts.  Four specialists are registered: `science_explainer`, `history_story`, `how_to_tutorial`, `product_review`.  The router classifies the user's topic request into one of those labels (or `unknown`) and dispatches.


In [ ]:
SPECIALISTS = {
    "science_explainer": """You write a 60-90 second video script that explains a scientific concept to a curious general audience.

Structure (use these exact headers, one blank line between sections):
[HOOK] — One sentence that frames the concept as a question or surprise.  About 10 seconds.
[BODY] — Three or four numbered beats that build the explanation.  Use exactly one concrete analogy.  Avoid jargon — if a technical term is unavoidable, define it inline.
[PAYOFF] — One sentence that resolves the hook and leaves the viewer with a surprising fact.

Tone: curious, precise, friendly.  No "hey everyone" preamble.""",

    "history_story": """You write a 60-90 second video script that tells the story of a historical event, figure, or era.

Structure (use these exact headers, one blank line between sections):
[COLD OPEN] — A single vivid image or moment that drops the viewer into the story.  About 10 seconds.
[STORY] — Three numbered beats arranged as setup, turn, consequence.  Name specific people, places, and dates.  Only include facts you are confident about.
[RESONANCE] — One sentence that connects the past event to something the viewer will recognize today.

Tone: narrative, evocative.  No "today on the channel" preamble.""",

    "how_to_tutorial": """You write a 60-90 second video script that teaches the viewer to do a specific practical task.

Structure (use these exact headers, one blank line between sections):
[WHAT YOU NEED] — A short bullet list of materials, tools, or prerequisites.
[STEPS] — A numbered list of five to seven steps.  Each step starts with an imperative verb ("Whisk the eggs", "Open the terminal").  Call out the most common mistake inline where it would happen.
[CHECK] — One sentence describing how the viewer knows they did it right.

Tone: direct, encouraging.  Address the viewer as "you".""",

    "product_review": """You write a 60-90 second video script that reviews a specific product, service, or piece of software.

Structure (use these exact headers, one blank line between sections):
[VERDICT FIRST] — One sentence with a clear recommendation ("buy", "skip", "wait for v2") and a price or availability anchor.
[WHAT WORKS] — Two or three concrete strengths.  Cite specific behavior, not vibes.
[WHAT DOESN'T] — Two or three concrete weaknesses.  Name the use case where each weakness matters.
[BUY IF / SKIP IF] — One sentence each: who it's for, who should pass.

Tone: opinionated but fair.  No "we'll see in this video" preamble.""",
}

ROUTER_SYSTEM = """You categorize a user's video-script request into exactly one of the labels below.

Labels and what they cover:
- science_explainer: explaining a scientific concept, natural phenomenon, or how something works (physics, biology, chemistry, math, engineering).
- history_story: a narrative about a historical event, figure, or era.
- how_to_tutorial: a step-by-step practical guide where the viewer follows along (cooking, DIY, software, fitness).
- product_review: an evaluation of a specific product, service, or piece of software with a clear recommendation.

Output ONLY the label, lowercase, exactly as written above.  No punctuation, no quotes, no explanation.

If the request fits more than one label, choose the one that best matches the request's *primary* intent — for example, a request to explain *why* a historical machine worked is science_explainer, while a request to tell the *story* of its inventor is history_story.

If none of the labels fit, output exactly: unknown"""

In [ ]:
def route(request: str) -> str:
    messages = []
    add_user_message(messages, request)
    return chat(messages, system_prompt=ROUTER_SYSTEM, temperature=0.0).strip().lower()

def generate_script(request: str) -> str:
    label = route(request)
    print(f"router → {label}\n")
    if label not in SPECIALISTS:
        return (
            f"No specialist registered for category '{label}'.  "
            f"Please rephrase the request as one of: {', '.join(SPECIALISTS)}."
        )
    messages = []
    add_user_message(messages, request)
    return chat(messages, system_prompt=SPECIALISTS[label], temperature=0.7)

## Demo 1 — a clear-cut request

A request that lands unambiguously in one specialist.  The router picks the category and the specialist produces a script whose structure (HOOK / BODY / PAYOFF) reflects the science-explainer prompt — *not* a generic "video script" template.

In [ ]:
print(generate_script("Explain why the sky is blue"))

## Demo 2 — an ambiguous request

The Apollo program is both a science story (engineering, physics, the LM ascent stage) and a history story (Cold War, JFK's speech, the people who built it).  The router has to make a judgment.  Read its choice and the resulting script — the structure it picks tells you which lens the router decided to apply.

In [ ]:
print(generate_script("Tell me about the Apollo program"))

## Demo 3 — an out-of-scope request

Asking for something none of the four specialists are equipped to handle.  A well-built router returns `unknown` rather than guessing — which is exactly what lets the calling code refuse gracefully instead of producing a confidently-wrong answer.

In [ ]:
print(generate_script("Translate this French poem to English: \"Demain, dès l'aube, à l'heure où blanchit la campagne...\""))

## Things to try

- Run the router on Haiku and the specialists on Sonnet by introducing a second `chat()` helper that takes a `model` argument.  Classification is almost always the cheaper task — a real production router pays Sonnet rates only for the specialist call.
- Replace the prompt-based router with a tool-use router: define one tool whose `category` parameter is an `enum` of the four labels, and have the model call it.  The API will enforce the enum on the structured side, which is more robust than parsing free-form text.
- Add a fifth specialist (e.g. `myth_or_folklore`) by adding one entry to `SPECIALISTS` and one line to `ROUTER_SYSTEM`.  Nothing else changes.  Contrast that with the diff that would be required to add the same capability to a single mega-prompt.
- Compose with Chaining: route once at the top, then within the chosen specialist run the three-step extract/enrich/compose chain from the previous section.  Routing and chaining compose naturally — routing picks the *which*, chaining controls the *how*.